In [1]:
# defining worker slots

class worker_slot:
    """
    -------------------------------------------------------------
    Description
    
        Slots to occupy worker objects
    -------------------------------------------------------------
    Attributes
    
        ID: str: A unique identifier for the slot.
        slotClass: str: Type of worker that can fill the slot
        slotSkill: str: Skill required to fill the slot
        occupier: class object: the worker object currently occupying the slot
        parent: class object: the company or area that holds the slot
    -------------------------------------------------------------
    Methods
    
        removeOccupier: sets occupier to None
        addOccupier: sets occupier to a worker class using the worker's ID
        updateParent: set the parent to a master class using the class' ID (not currently used)
    -------------------------------------------------------------
    """
    
    def __init__(
        self,
        ID, 
        slotClass, 
        slotSkill, 
        occupier = None,
        parent = None 
    ):
        self.__ID = ID
        self.__slotClass = slotClass
        self.__slotSkill = slotSkill
        self._occupier = occupier
        self.__parent = parent
    
    def removeOccupier(self) -> None:
        """sets occupier to None"""
        self.occupier = None
        
    def addOccupier(self, workerID) -> None:
        """sets occupier attribute to a worker class using the worker's ID"""
        self.occupier = workerID

        
class Company:
    # parent company class
    def __init__(self, ID, name, owner, cost, product, baseProduction, workerSlots = (), strike = False, parentSlot = None):
        self.__ID = ID
        self.__name = name
        self._owner = owner
        self.__cost = cost
        self.__product = product
        self.__baseProduction = baseProduction
        self.__production = baseProduction
        self.__workerSlots = workerSlots
        self._strike = strike
        self._parentSlot = parentSlot
        
    def createWorkerSlot(self, slotClass, slotSkill):
        """create a new workerSlot class instance as a child of this instance"""
        existingSlots = list(self.workerSlots)  # Convert the tuple to a list
        IDcounter = len(existingSlots)  # Generate a new ID for the worker slot
        newSlot = WorkerSlot(self.ID + "Slot" + str(IDcounter), slotClass, slotSkill, None, self)
        existingSlots.append(newSlot)  # Add the new WorkerSlot to the list
        self.workerSlots = tuple(existingSlots)  # Convert the list back to a tuple
     
    def setParentSlot(self, parentSlotID):
        """sets parent attribute to a company or area class using the new parent's ID"""
        self.parentSlot = parentSlotID
        
    def checkOperational(self):
        """check if operational"""
        if len(self.workerSlots) > 0:
                if self.workerSlots[0].occupier is not None and not self.strike:
                    self.operational = True
                else:
                    self.operational = False
        else:
            self.operational = True
        
        
class PublicSectorCompany(Company):
    def __init__(self, ID, name, owner, cost, product, baseProduction, salaryOptions = None, salaryLevel = 0, workerSlots = (), strike = False):
        
        #use parent init method
        super().__init__(ID, name, owner, cost, product, baseProduction, workerSlots, strike)
        self.salaryOptions = salaryOptions
        #update salary figure
        self.setSalaryLevel(salaryLevel)        
        
    def beginStrike(self):
        self.strike = True
        
    def endStrike(self):
        self.strike = False
        
    def setSalaryLevel(self, newSalaryLevel):
        self.salaryLevel = newSalaryLevel
        self.salary = salaryOptions["L" + str(newSalaryLevel)] if salaryOptions else None
                
        

class CapitalistCompany(PublicSectorCompany):
    
    def __init__(self, ID, name, owner, cost, product, baseProduction, salaryOptions = None, salaryLevel = 0, workerSlots = (), upgradeValue = 0, upgraded = False, strike = False):
        # call parent init method
        super().__init__(ID, name, owner, cost, product, baseProduction, salaryOptions, salaryLevel, workerSlots, strike)
        self.upgradeValue = upgradeValue
        self.upgraded = upgraded        

        # check upgradeable
        self.upgradeable = upgradeValue > 0
            
        self.checkOperational()
        self.updateProduction()
  

    def checkOperational(self):
        # check if operational
        if len(self.workerSlots) > 0:
                if self.workerSlots[0].occupier is not None and not self.strike:
                    self.operational = True
                else:
                    self.operational = False
        else:
            self.operational = True
                      
    def updateProduction(self):
        # update production value - will return 0 if company is disabled
        self.production = (self.upgraded * self.upgradeValue + self.baseProduction) * self.operational
     
    def upgrade(self):
        if not upgradeable or owner != 'Capitalists':
            print("Error - " + self.ID + " cannot be upgraded")
        else:
            self.upgraded = True
            self.updateProduction()
            
            
class MiddleClassCompany(CapitalistCompany):
    
    def __init__(self, ID, name, owner, cost, product, baseProduction, salaryOptions = None, salaryLevel = 0, workerSlots = (), upgradeValue = 0, upgraded = False, strike = False):
        # call parent init method
        super().__init__(ID, name, owner, cost, product, baseProduction, salaryOptions, salaryLevel, workerSlots, upgradeValue, upgraded, strike)
  

    def checkOperational(self):
        # check if operational
        if len(self.workerSlots) > 0:
                if self.workerSlots[0].occupier is not None and not self.strike:
                    self.operational = True
                else:
                    self.operational = False
        else:
            self.operational = True
                      
    def updateProduction(self):
        # update production value - will return 0 if company is disabled
        self.production = (self.upgraded * self.upgradeValue + self.baseProduction) * self.operational
        ## UPDATE FOR MIDDLE CLASS WORKERS           
       
            
        


In [18]:
class workerSlot.desc

SyntaxError: invalid syntax (2604540828.py, line 1)

In [15]:
import pandas as pd
import os
cwd = os.getcwd()
companies_df = pd.read_csv(cwd+"\Hegemony Companies.csv")

In [13]:
# read in company info and create company decks
i = 0
companyDict = {}
for index, row in companies_df.iterrows():
    
    # update salary range
    if companies_df.isnull().loc[index,'L1']:
        salaryOptions = {
            'L1':row['L1'],
            'L2':row['L2'],
            'L3':row['L3'],
        }
    else:
        salaryOptions = None
        
    #check upgradeable
    if companies_df.isnull().loc[index,'Upgrade Value']:
        upgradeValue = row['Upgrade Value']
    else:
        upgradeValue = 0
        
    if row['Owner'] == 'Working Class': 
        companyDict['Company' + str(i)] = Company('Company' + str(i), 
                                             row['Name'], 
                                             row['Owner'], 
                                             row['Cost'], 
                                             row['Product'], 
                                             row['Base Production']
                                            )
        
    elif row['Owner'] == "State":
        companyDict['Company' + str(i)] = PublicSectorCompany('Company' + str(i), 
                                                         row['Name'], 
                                                         row['Owner'], 
                                                         row['Cost'], 
                                                         row['Product'], 
                                                         row['Base Production'],
                                                         salaryOptions,
                                                         2,
                                                         () # we create workerslots later
                                                        )
    elif row['Owner'] == 'Capitalists':
        companyDict['Company' + str(i)] = CapitalistCompany('Company' + str(i),
                                                         row['Name'], 
                                                         row['Owner'], 
                                                         row['Cost'], 
                                                         row['Product'], 
                                                         row['Base Production'],
                                                         salaryOptions,
                                                         2,
                                                          (),
                                                          upgradeValue
                                                         )
    elif row['Owner'] == "Middle Class":
        companyDict['Company' + str(i)] = MiddleClassCompany('Company' + str(i),
                                                         row['Name'], 
                                                         row['Owner'], 
                                                         row['Cost'], 
                                                         row['Product'], 
                                                         row['Base Production'],
                                                         salaryOptions,
                                                         2,
                                                          (),
                                                          upgradeValue
                                                       )
    else:
        print('Error - Company owner not recognised')
    
    
    # create worker slots
    if not companies_df.isnull().loc[index,'Class1']:    
        companyDict['Company' + str(i)].createWorkerSlot(row['Class1'], row['Skill1'])
    if not companies_df.isnull().loc[index,'Class2']:    
        companyDict['Company' + str(i)].createWorkerSlot(row['Class2'], row['Skill2'])
    if not companies_df.isnull().loc[index,'Class3']:    
        companyDict['Company' + str(i)].createWorkerSlot(row['Class3'], row['Skill3'])
    i += 1
        

In [5]:
class TradeUnion:
    def __init__(self, industry):
        self.ID = "TradeUnion" + industry
        self.industry = industry
        self.workerSlots = WorkerSlot("TradeUnion" + industry + "Slot", "Working Class", industry, None, "TradeUnion" + industry)
        
tradeUnions = {"Agriculture": TradeUnion("Agriculture"), 
               "Luxury": TradeUnion("Luxury"),
               "Healthcare": TradeUnion("Healthcare"),
               "Education": TradeUnion("Education"),
               "Media": TradeUnion("Media")}

In [6]:
class Worker:
    def __init__(self, ID, workerClass, workerSkill, assignment = "off-board", committed = False):
        self.ID = ID
        self.workerClass = workerClass
        self.workerSkill = workerSkill
        self.assignment = assignment
        self.committed = committed
        
    def updateAssignment(self, workerSlot):
        self.assignment = workerSlot
        
    def commitWorker(self):
        self.committed = True
        
    def uncommitWorker(self):
        self.committed = False
        
        
# defining workers
def createWorkerPool(): 
    # this resets all workers, but could create duplicates if not done alongside a reset of all worker slots
    
    workerPool = []
    
    # creating working class workers
    i = 1
    for industry in list(tradeUnions.keys()):
        for j in range(5):            
            workerPool.append(Worker("WCWorker" + str(i), "Working Class", industry, "off-board"))
            i += 1
    while i < 49:
        workerPool.append(Worker("WCWorker" + str(i), "Working Class", "Unskilled", "off-board"))
        i += 1

    # creating middle class workers
    i = 1
    for industry in list(tradeUnions.keys()):
        for j in range(5):
            workerPool.append(Worker("MCWorker" + str(i), "Middle Class", industry, "off-board"))
            i += 1
    while i < 43:
        workerPool.append(Worker("MCWorker" + str(i), "Middle Class", "Unskilled", "off-board"))
        i += 1

    return workerPool


### Main ###


offMapWorkers = createWorkerPool()

In [7]:
# creating unemployment worker slots
unemployedWorkers = []
for i in range(len(offMapWorkers)):
    unemployedWorkers.append(WorkerSlot("unemployed" + str(i), "Any", "Any", None, "Unemployed Workers"))

In [8]:
def findEligibleWorkers(desiredClass, desiredSkill, sourcePool):
    
    desiredSkill = desiredSkill if desiredSkill != 'Any' else 'Unskilled'
    closeMatches = []
    for worker in sourcePool:
        if worker.workerClass == desiredClass and worker.workerSkill == desiredSkill:
            return worker
        elif worker.workerClass == desiredClass and desiredSkill == 'Unskilled':
            closeMatches.append(worker)
    
    if len(closeMatches) == 0:
        print("Error selecting worker - no matches found")
    return closeMatches[0]
        

In [9]:
def transferWorker(sourceSlot, destinationSlot):
    # moves a worker and updates attributes of the worker and slots
    worker = source.occupier
    
    # update worker attributes
    worker.updateAssignment(destinationSlot)
    
    #update source slot attributes
    sourceSlot.removeOccupier()
    
    #update destination slot attributes
    destinationSlot.addOccupier(worker)
    
    

In [10]:
def birthWorker(workerClass, workerSkill):
    # find eligible worker
    try:
        worker = findEligibleWorkers(workerClass, workerSkill, offMapWorkers)
    except:
        print('Unable to spawn new worker - no eligible candidates found')
        return
    
    for i in range()
    # update worker attributes
    worker.updateAssignment()
    
    # update unemployed worker slot attributes
        
    
    
    

SyntaxError: expected ':' (3699977974.py, line 9)

In [ ]:
offMapWorkers

In [ ]:
transferWorker()